# Phase 3, Task 1 — Read the Confusion Matrix & Turn the Threshold Dial

Phase 2 answered *"is the model good?"*. Phase 3 answers the real SOC question: **"at what sensitivity do we run it?"**

Until now you've accepted whatever `.predict()` returned. But `.predict()` secretly means *"flag as attack only if the model is ≥ 50% sure."* **50% is an arbitrary default, not a law** — it's a dial you own. A detector that must never miss a breach flags at a *lower* bar (catch more, tolerate more false alarms); one drowning analysts in noise raises the bar.

To tune that dial you need two things:
1. The **confusion matrix** — the 4 raw counts behind every score, so you see the exact mistakes.
2. The model's **probabilities** (`predict_proba`), not just its 0/1 verdict, so you can re-decide at any cutoff.

**Vocabulary (define-as-we-go):**
- **TP** attack, caught · **FP** benign, wrongly flagged = *false alarm* · **FN** attack, missed = *the breach that walked in* · **TN** benign, correctly ignored.
- **Precision** = TP / (TP+FP) — when it flags, how often it's right. **Recall** = TP / (TP+FN) — of all real attacks, how many it caught. Both are just ratios of those 4 numbers.
- **False-positive cost / alert fatigue** = every FP burns analyst time; too many and the team starts ignoring alerts, so real ones get missed too. You can't max precision AND recall — you *choose* the balance.

## Step 0 — Rebuild the substrate (reuse your Task 5 work)
We need a model with **real misses** to analyze — a perfect 1.00/1.00 model has nothing to tune. So reuse the **held-out `model2` from Task 5's stretch**: trained on the 7 days *minus* PortScan. Its ~31k missed PortScans (plus any borderline cases) are our raw material.

**Your job:** reproduce, from your Task 5 notebook, everything up to and including `model2`:
- load + `load_clean` the 7 day CSVs → `pd.concat` into one `df` (keep `Label`)
- build `is_attack`, `X`, `y`; stratified 80/20 split, **also capturing `label_test`** (the attack name per test row) and `lab_train`
- `mask = lab_train != 'PortScan'` → train `model2` on `X_train[mask], y_train[mask]`

**Hint:** you already wrote all of this — copy your own cells over. End this step with `model2`, `X_test`, `y_test`, `label_test` all defined. (Sanity check: `y_test.mean()` ≈ 0.22 attack rate.)

In [ ]:
# TODO: reproduce Task 5 up to model2 (7 days minus PortScan). End with model2, X_test, y_test, label_test defined.


## Step 1 — Get probabilities, not labels
`.predict()` throws away *how sure* the model is. `predict_proba` keeps it — that confidence is exactly what the threshold dial acts on.

**Your job:** get the attack-probability for every test row and eyeball a few.

**Hints:**
- `proba = model2.predict_proba(X_test)[:, 1]` — `predict_proba` returns TWO columns: `[:, 0]` = P(benign), `[:, 1]` = P(attack). You want column 1.
- Look at `proba[:20]`. Notice most values sit near 0 or near 1 — random forests are confident. Keep that in mind for Step 5.

In [ ]:
# TODO: proba = P(attack) for each test row; eyeball the first 20


## Step 2 — The confusion matrix at the default 0.50
The confusion matrix is the 4 raw counts that *every* score is built from. See the mistakes directly.

**Your job:** turn `proba` into 0/1 at the default cutoff and print the confusion matrix.

**Hints:**
- `y_pred = (proba >= 0.5).astype(int)`
- `from sklearn.metrics import confusion_matrix` → `confusion_matrix(y_test, y_pred)` prints `[[TN, FP], [FN, TP]]` (rows = actual, cols = predicted, for labels 0 then 1).
- Sanity-check: recall = TP / (TP + FN). Does it match the ~0.71 attack recall you got in the Task 5 stretch?

In [ ]:
# TODO: y_pred at 0.50; print confusion_matrix(y_test, y_pred)


### ✍️ Label the four numbers (in your own words)
Fill these in from your matrix above — this is the vocabulary sticking:

- **TN** (benign, correctly ignored) = ?
- **FP** (false alarms — benign flagged as attack) = ?
- **FN** (missed attacks — the breaches that walked in) = ?
- **TP** (attacks caught) = ?

Which single number is the one a SOC fears most, and why?

## Step 3 — Turn the dial
Now move the cutoff and watch precision and recall trade against each other.

**Your job:** for each threshold `t` in `[0.1, 0.3, 0.5, 0.7, 0.9]`, recompute predictions and print the attack-class **recall AND precision**.

**Hints:**
- A small loop beats 5 copy-pasted cells: `for t in [...]:` then `y_t = (proba >= t).astype(int)`.
- `from sklearn.metrics import recall_score, precision_score` → `recall_score(y_test, y_t)`, `precision_score(y_test, y_t)`.
- Print like `f"t={t}: recall={r:.3f} precision={p:.3f}"`.
- **Watch the direction:** as `t` drops, which one climbs and which one falls? Say it out loud before moving on.

In [ ]:
# TODO: loop thresholds 0.1..0.9; print attack recall + precision at each


## Step 4 — Draw the trade-off (the PR curve)
Those 5 points are a sample of a continuous curve. The full precision-recall curve *is* the menu of every operating point you could choose.

**Your job:** compute and plot the precision-recall curve.

**Hints:**
- `from sklearn.metrics import precision_recall_curve` → `precision, recall, thr = precision_recall_curve(y_test, proba)`.
- `import matplotlib.pyplot as plt` → `plt.plot(recall, precision)`; label the axes (x = recall, y = precision), add a title.
- Where the curve stays high-and-flat you get recall almost for free; where it suddenly drops is the price of chasing the last few attacks.

In [ ]:
# TODO: precision_recall_curve(y_test, proba); plot recall (x) vs precision (y)


## Step 5 — Make an engineering decision (and state its price)
This is the actual job: pick an operating point on purpose, in numbers.

**Your job:** set a goal — e.g. *"catch ≥ 90% of attacks"* — find the **lowest threshold** that hits it, then report the **cost**: how many more false alarms (FP) than the default 0.50? Is the trade worth it?

**Hints:**
- From the `precision_recall_curve` arrays, find where `recall >= 0.90` and read off the matching threshold (`thr` is aligned to `precision[:-1]`/`recall[:-1]`).
- Then rebuild `y_pred` at that threshold and compare its confusion matrix (esp. the FP count) to Step 2's.
- There is no single right answer — the skill is *stating the trade-off in numbers*, not finding a magic cutoff.

In [ ]:
# TODO: find lowest threshold with recall >= target; report its FP cost vs the 0.50 default


## Step 6 — Interpret (for LEARNING_LOG, your own words)
You'll likely see one of two honest outcomes — **both are the lesson**:
1. For *borderline* errors, lowering the threshold recovers real attacks at the cost of more false alarms — **the dial works.**
2. For the held-out **PortScan**, the model is *confidently* wrong (its `proba` ≈ 0), so even a very low threshold barely recovers it — **threshold tuning fixes borderline cases, not blind spots.**

Answer:
1. Which way did recall and precision move as you dropped the threshold, and why?
2. Did lowering the threshold recover much of the missed PortScan? What does that tell you about the *kind* of error threshold tuning can and can't fix?
3. In one sentence: what does a detection engineer actually decide when they "set the threshold"?

---
### ✅ You pass Phase 3 Task 1 when:
1. You printed a **confusion matrix** and labelled its four cells (TP/FP/FN/TN) in your own words.
2. You produced a **precision/recall-vs-threshold** table and a **PR curve**.
3. You chose an operating threshold for a stated goal and reported its **false-positive cost in numbers**.
4. You can explain which misses threshold tuning fixes (borderline) and which it can't (confident blind spots).

Paste me your confusion matrix + threshold table + your Step 6 answers, and I'll review.